# Import Các Thư Viện

In [1]:
import matplotlib
matplotlib.use('Agg')  # Sử dụng backend không yêu cầu GUI

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import numpy as np
from scipy.stats import randint, uniform
import joblib


# Hàm Tiền Xử Lý 'distance_from_expressway'

In [2]:
def preprocess_distance(df):
    """
    Chuyển đổi cột 'distance_from_expressway' từ dạng chuỗi sang số.
    Xử lý các định dạng như '101-150m', '<=50m', '>500m', và '300m'.
    """
    def convert_distance(value):
        if isinstance(value, str):
            value = value.replace('m', '').strip()
            try:
                if '-' in value:
                    # Xử lý khoảng cách, ví dụ: '101-150m'
                    parts = value.split('-')
                    low = float(parts[0])
                    high = float(parts[1])
                    return (low + high) / 2
                elif value.startswith('<='):
                    # Xử lý '<=50m'
                    num = float(value.replace('<=', '').strip())
                    return num
                elif value.startswith('>'):
                    # Xử lý '>500m'
                    num = float(value.replace('>', '').strip())
                    return num
                else:
                    # Xử lý giá trị đơn lẻ, ví dụ: '300m'
                    return float(value)
            except:
                return np.nan  # Trả về NaN nếu không thể chuyển đổi
        elif np.issubdtype(type(value), np.number):
            return value
        else:
            return np.nan  # Trả về NaN cho các loại dữ liệu khác

    df['distance_from_expressway'] = df['distance_from_expressway'].apply(convert_distance)
    return df


# Hàm Tính Tỷ Lệ Dự Đoán Trong Khoảng Sai Số Nhất Định

In [3]:
def calculate_accuracy_percentage(y_true, y_pred, tolerance=0.10):
    """
    Tính tỷ lệ phần trăm dự đoán nằm trong ±tolerance (mặc định 10%) so với giá trị thực tế.
    """
    lower_bound = y_true * (1 - tolerance)
    upper_bound = y_true * (1 + tolerance)
    within_tolerance = ((y_pred >= lower_bound) & (y_pred <= upper_bound))
    accuracy_percentage = within_tolerance.mean() * 100
    return accuracy_percentage


# Load Data và Khám Phá Dữ Liệu (EDA)

In [4]:
# Load dataset
data = pd.read_csv('resale.csv')

# Step 1: EDA
print("Thông tin Dataset:")
data.info()

print("\nThống kê mô tả:")
print(data.describe())

print("\nGiá trị thiếu:")
print(data.isnull().sum())


Thông tin Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220971 entries, 0 to 220970
Data columns (total 11 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   year                      220971 non-null  int64  
 1   town                      220971 non-null  object 
 2   flat_type                 220971 non-null  object 
 3   block                     220971 non-null  object 
 4   street_name               220971 non-null  object 
 5   storey_range              220971 non-null  object 
 6   floor_area_sqm            220971 non-null  float64
 7   remaining_lease_years     220971 non-null  int64  
 8   resale_price              220971 non-null  float64
 9   storey_range_category     220971 non-null  object 
 10  distance_from_expressway  220971 non-null  object 
dtypes: float64(2), int64(2), object(7)
memory usage: 18.5+ MB

Thống kê mô tả:
                year  floor_area_sqm  remaining_lease_years  r

## Vẽ Biểu Đồ Phân Phối của Biến Mục Tiêu

In [5]:
# Plot distribution of target variable
plt.figure(figsize=(8,6))
sns.histplot(data['resale_price'], kde=True, bins=30)
plt.title("Phân phối Giá Bán Lại")
plt.xlabel("Giá Bán Lại")
plt.ylabel("Tần suất")
plt.savefig('distribution_resale_price.png')  # Lưu biểu đồ thay vì hiển thị
plt.close()


## Heatmap Ma Trận Tương Quan

In [6]:
# Correlation heatmap for numerical features
numerical_features_eda = ['year', 'floor_area_sqm', 'remaining_lease_years', 'resale_price']
correlation_matrix = data[numerical_features_eda].corr()
plt.figure(figsize=(8,6))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm")
plt.title("Ma Trận Tương Quan")
plt.savefig('correlation_heatmap.png')  # Lưu biểu đồ thay vì hiển thị
plt.close()


# Tiền Xử Lý Dữ Liệu

## Xác Định Biến Mục Tiêu và Đặc Trưng

In [7]:
# Define target and features
target = 'resale_price'
features = ['year', 'town', 'flat_type', 'floor_area_sqm',
            'remaining_lease_years', 'distance_from_expressway']


## Loại Bỏ Các Hàng Có Giá Trị Thiếu Trong Biến Mục Tiêu

In [8]:
# Drop rows with missing target values
data = data.dropna(subset=[target])


## Phân Chia Dữ Liệu Thành Tập Huấn Luyện và Kiểm Tra

In [9]:
# Testing (100% of the data)
X = data[features].copy()
y = data[target]

# Preprocess 'distance_from_expressway' with updated function
X = preprocess_distance(X)

# Kiểm tra lại giá trị NaN sau khi chuyển đổi
print("\nGiá trị NaN trong 'distance_from_expressway' sau khi chuyển đổi:", X['distance_from_expressway'].isnull().sum())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



Giá trị NaN trong 'distance_from_expressway' sau khi chuyển đổi: 0


## Định Nghĩa Các Đặc Trưng Phân Loại và Số

In [10]:
# Define categorical and numerical features
categorical_features = ['town', 'flat_type']
numerical_features = ['year', 'floor_area_sqm', 'remaining_lease_years', 'distance_from_expressway']


## Xây Dựng Các Pipeline Tiền Xử Lý

In [12]:
# Preprocessing cho dữ liệu số: Impute và Scale
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Preprocessing cho dữ liệu phân loại: Impute và One-Hot Encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Kết hợp các bước tiền xử lý
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


# Xây Dựng Model

## Xây Dựng Pipeline với Random Forest

In [13]:
# Step 3: Build ML Model
# Tạo pipeline với Random Forest
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42, n_jobs=-1))
])


## Định Nghĩa Phân Phối Siêu Tham Số cho RandomizedSearchCV

In [14]:
# Định nghĩa phân phối siêu tham số cho RandomizedSearchCV
param_dist = {
    'regressor__n_estimators': randint(70, 90),         # Tập trung quanh giá trị 79
    'regressor__max_depth': [None],                     # Giá trị cố định None
    'regressor__min_samples_split': randint(8, 11),     # Tập trung quanh giá trị 9
    'regressor__min_samples_leaf': randint(1, 3)        # Tập trung quanh giá trị 2
}


## Khởi Tạo và Huấn Luyện RandomizedSearchCV

In [15]:
# Initialize RandomizedSearchCV
random_search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_dist,
    n_iter=20,               # Số lượng tổ hợp thử nghiệm ngẫu nhiên
    cv=3,                    # Số fold trong cross-validation
    scoring='r2',
    n_jobs=-1,
    verbose=2,
    random_state=42,
    error_score='raise'     # Đặt để lỗi không bị bỏ qua
)

# Huấn luyện mô hình với Randomized Search
try:
    random_search.fit(X_train, y_train)
except Exception as e:
    print("Error during Randomized Search:", e)
    raise

# In ra siêu tham số tốt nhất
print("\nSiêu tham số tốt nhất:", random_search.best_params_)


Fitting 3 folds for each of 20 candidates, totalling 60 fits

Siêu tham số tốt nhất: {'regressor__max_depth': None, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 8, 'regressor__n_estimators': 84}


# Đánh Giá Mô Hình

## Dự Đoán và Tính Toán Các Chỉ Số Đánh Giá

In [16]:
# Step 4: Evaluation
# Dự đoán trên tập huấn luyện và kiểm tra
y_pred_train = random_search.predict(X_train)
y_pred_test = random_search.predict(X_test)

# Tính các chỉ số đánh giá
mse_train = mean_squared_error(y_train, y_pred_train)
mse_test = mean_squared_error(y_test, y_pred_test)
rmse_train = np.sqrt(mse_train)
rmse_test = np.sqrt(mse_test)
mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test = mean_absolute_error(y_test, y_pred_test)
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)


## In Kết Quả Đánh Giá

In [17]:
# In kết quả đánh giá
print("\nĐánh giá mô hình:")
print(f"Train Mean Squared Error: {mse_train:.2f}")
print(f"Test Mean Squared Error: {mse_test:.2f}")
print(f"Train Root Mean Squared Error: {rmse_train:.2f}")
print(f"Test Root Mean Squared Error: {rmse_test:.2f}")
print(f"Train Mean Absolute Error: {mae_train:.2f}")
print(f"Test Mean Absolute Error: {mae_test:.2f}")
print(f"Train R-squared: {r2_train:.4f}")
print(f"Test R-squared: {r2_test:.4f}")

# Kiểm tra overfitting
if r2_train - r2_test > 0.1:
    print("Mô hình có thể đang bị overfitting. Hãy xem xét điều chỉnh thêm các siêu tham số hoặc đơn giản hóa mô hình.")
else:
    print("Mô hình không có dấu hiệu bị overfitting.")



Đánh giá mô hình:
Train Mean Squared Error: 1050546782.36
Test Mean Squared Error: 1596393503.17
Train Root Mean Squared Error: 32412.14
Test Root Mean Squared Error: 39954.89
Train Mean Absolute Error: 23320.37
Test Mean Absolute Error: 28296.03
Train R-squared: 0.9637
Test R-squared: 0.9452
Mô hình không có dấu hiệu bị overfitting.


## So Sánh Giá Trên Tập Kiểm Tra

In [18]:
# Output actual vs predicted prices for the test set
results = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred_test
}).reset_index(drop=True)

print("\nMột số kết quả thực tế và dự đoán:")
print(results.head())

# Plot Actual vs Predicted
plt.figure(figsize=(8,6))
sns.scatterplot(x='Actual', y='Predicted', data=results, alpha=0.6)
plt.plot([results['Actual'].min(), results['Actual'].max()],
         [results['Actual'].min(), results['Actual'].max()],
         color='red', linestyle='--')
plt.xlabel("Giá Bán Lại Thực Tế")
plt.ylabel("Giá Bán Lại Dự Đoán")
plt.title("So Sánh Giá Bán Lại Thực Tế và Dự Đoán")
plt.savefig('actual_vs_predicted.png')  # Lưu biểu đồ thay vì hiển thị
plt.close()



Một số kết quả thực tế và dự đoán:
     Actual      Predicted
0  805000.0  780279.874937
1  510000.0  490474.812611
2  472000.0  468041.629188
3  358000.0  345266.728112
4  480000.0  477247.547712


# Đánh Giá Bổ Sung

## Phân Tích Phần Trăm Dự Đoán Chính Xác

In [19]:
# Step 5: Additional Evaluation

## 1. Phân Tích Phần Trăm Dự Đoán Chính Xác
accuracy_10 = calculate_accuracy_percentage(y_test, y_pred_test, tolerance=0.10)
accuracy_5 = calculate_accuracy_percentage(y_test, y_pred_test, tolerance=0.05)

print(f"\nTỷ lệ dự đoán trong ±10%: {accuracy_10:.2f}%")
print(f"Tỷ lệ dự đoán trong ±5%: {accuracy_5:.2f}%")



Tỷ lệ dự đoán trong ±10%: 84.05%
Tỷ lệ dự đoán trong ±5%: 54.59%


## Phân Tích Residual (Sai Số Dự Đoán)

In [20]:
## 2. Phân Tích Residual (Sai Số Dự Đoán)
# Tính residual
residuals = y_test - y_pred_test

# Phân phối residual
plt.figure(figsize=(8,6))
sns.histplot(residuals, kde=True, bins=30, color='purple')
plt.title("Phân phối Residuals")
plt.xlabel("Residual")
plt.ylabel("Tần suất")
plt.savefig('residuals_distribution.png')
plt.close()

# Plot residuals vs predicted values
plt.figure(figsize=(8,6))
sns.scatterplot(x=y_pred_test, y=residuals, alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Giá Bán Lại Dự Đoán")
plt.ylabel("Residual")
plt.title("Residuals vs Predicted Values")
plt.savefig('residuals_vs_predicted.png')
plt.close()


## Hiển Thị Đường 45 Độ

In [21]:
## 3. Hiển Thị Đường 45 Độ
plt.figure(figsize=(8,6))
sns.scatterplot(x='Actual', y='Predicted', data=results, alpha=0.6)
plt.plot([results['Actual'].min(), results['Actual'].max()],
         [results['Actual'].min(), results['Actual'].max()],
         color='red', linestyle='--', label='y = x')
plt.xlabel("Giá Bán Lại Thực Tế")
plt.ylabel("Giá Bán Lại Dự Đoán")
plt.title("So Sánh Giá Bán Lại Thực Tế và Dự Đoán với Đường y = x")
plt.legend()
plt.savefig('actual_vs_predicted_with_line.png')
plt.close()


## Kiểm Tra Độ Chính Xác Theo Phân Nhóm (Theo Khu Vực)

In [22]:
## 4. Kiểm Tra Độ Chính Xác Theo Phân Nhóm
# Tính độ chính xác theo từng khu vực (town)
# Reset index cho y_test và đồng bộ dữ liệu
y_test = y_test.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

# Sao chép kết quả và thêm cột 'Town'
town_accuracy = results.copy()
town_accuracy['Town'] = X_test['town'].values

# Tính toán cột 'Within_10%'
town_accuracy['Within_10%'] = (town_accuracy['Predicted'] >= y_test * 0.90) & (town_accuracy['Predicted'] <= y_test * 1.10)

# Tính độ chính xác theo từng khu vực (Town)
accuracy_by_town = town_accuracy.groupby('Town')['Within_10%'].mean() * 100

print("\nĐộ chính xác trong ±10% theo từng khu vực:")
print(accuracy_by_town.sort_values(ascending=False))



Độ chính xác trong ±10% theo từng khu vực:
Town
WOODLANDS          90.337859
SEMBAWANG          89.734816
CHOA CHU KANG      88.749396
PUNGGOL            88.591502
TAMPINES           87.832828
JURONG WEST        86.910149
PASIR RIS          86.838006
BUKIT TIMAH        86.440678
BUKIT BATOK        85.852273
YISHUN             85.426621
BEDOK              85.319950
SENGKANG           84.576414
CENTRAL AREA       82.115869
KALLANG/WHAMPOA    81.557678
JURONG EAST        81.473214
GEYLANG            81.267474
HOUGANG            80.568079
BUKIT MERAH        80.210158
BUKIT PANJANG      79.940299
ANG MO KIO         79.862579
TOA PAYOH          78.801498
QUEENSTOWN         76.749799
BISHAN             76.418663
CLEMENTI           75.000000
MARINE PARADE      73.913043
SERANGOON          67.919799
Name: Within_10%, dtype: float64


## Tóm Tắt Các Chỉ Số Đánh Giá

In [23]:
## 5. Tóm tắt các chỉ số đánh giá
evaluation_metrics = pd.DataFrame({
    'Metric': ['Mean Squared Error', 'Root Mean Squared Error', 'Mean Absolute Error', 'R-squared'],
    'Train': [mse_train, rmse_train, mae_train, r2_train],
    'Test': [mse_test, rmse_test, mae_test, r2_test]
})

print("\nTóm tắt các chỉ số đánh giá:")
print(evaluation_metrics)



Tóm tắt các chỉ số đánh giá:
                    Metric         Train          Test
0       Mean Squared Error  1.050547e+09  1.596394e+09
1  Root Mean Squared Error  3.241214e+04  3.995489e+04
2      Mean Absolute Error  2.332037e+04  2.829603e+04
3                R-squared  9.637371e-01  9.452477e-01


# Lưu Kết Quả Đánh Giá

In [24]:
# Step 6: Lưu kết quả đánh giá
evaluation_metrics.to_csv('model_evaluation_metrics.csv', index=False)
accuracy_by_town.to_csv('accuracy_by_town.csv', header=True)


# Phân Tích Tầm Quan Trọng của Các Đặc Trưng (Feature Importance)


In [25]:
## 6. Feature Importance
# Lấy mô hình tốt nhất từ RandomizedSearchCV
best_rf_model = random_search.best_estimator_.named_steps['regressor']

# Lấy tầm quan trọng của các đặc trưng
# Lấy tên các đặc trưng sau khi One-Hot Encoding
onehot_features = random_search.best_estimator_.named_steps['preprocessor'].transformers_[1][1].named_steps['onehot'].get_feature_names_out(categorical_features)
feature_names = numerical_features + list(onehot_features)

feature_importances = best_rf_model.feature_importances_

# Tạo DataFrame để hiển thị
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

# Plot tầm quan trọng của các đặc trưng
plt.figure(figsize=(10,8))
sns.barplot(x='Importance', y='Feature', data=importance_df)
plt.title("Tầm Quan Trọng của Các Đặc Trưng")
plt.xlabel("Tầm Quan Trọng")
plt.ylabel("Đặc Trưng")
plt.tight_layout()
plt.savefig('feature_importances.png')
plt.close()

print("\nTầm quan trọng của các đặc trưng đã được lưu vào 'feature_importances.png'.")



Tầm quan trọng của các đặc trưng đã được lưu vào 'feature_importances.png'.


# Đánh Giá Mô Hình Bằng Cross-Validation

In [26]:
## 7. Cross-Validation
# Đánh giá mô hình bằng cross-validation
cv_scores = cross_val_score(random_search.best_estimator_, X, y, cv=5, scoring='r2')
print("\nCross-Validation R-squared scores:", cv_scores)
print(f"Trung bình Cross-Validation R-squared: {cv_scores.mean():.4f}")

# Lưu kết quả Cross-Validation
cv_results = pd.DataFrame({
    'Fold': range(1, 6),
    'R-squared': cv_scores
})
cv_results.to_csv('cross_validation_results.csv', index=False)



Cross-Validation R-squared scores: [0.9064923  0.87479538 0.85557881 0.85323053 0.91379493]
Trung bình Cross-Validation R-squared: 0.8808


# Lưu Trữ Mô Hình Máy Học

In [27]:
# Step 7: Lưu mô hình
joblib.dump(random_search.best_estimator_, 'random_forest_resale_price_model.joblib')
print("\nMô hình đã được lưu vào 'random_forest_resale_price_model.joblib'.")



Mô hình đã được lưu vào 'random_forest_resale_price_model.joblib'.
